# AOU-4 — 4-check validation harness

Phase M3 / Wave 2. RESEARCH Validation Architecture §; AOU-LD-PIPELINE.md §9.

Inputs: 13 .npz files (10 AFR + 3 EUR) in workspace bucket from Wave 2 dev fire (AOU-2 output) + 2 BlockMatrix-shard dirs for HLA + 8p23 (Path A.3 per RESEARCH Q5; D-M3-09).

Outputs: 4 validation-output directories at `.planning/phases/m3-aou-afr-ld-panel-build/validation/` — heatmaps + invariants TSV (Check 1), per-region pearson_r_by_maf_bin.tsv (Check 2), susie_fit.rds + summary (Check 3), yield_table.tsv + maf_drop.tsv (Check 4) + sensitivity_cohort_r.tsv (D-M3-07).

Pass thresholds (RESEARCH Validation Architecture §):
- Check 1: visual block-boundary alignment + ≥ 0.85 pixel correlation (stretch goal); FTO 16q12 lead **rs1558902** ± 500 kb (Locke 2015) and SORT1 1p13 lead **rs12740374** ± 500 kb (Teslovich 2010).
- Check 2: mean Pearson r ≥ 0.97 for MAF ≥ 0.05; secondary ≥ 0.90 for MAF 0.01-0.05.
- Check 3: SuSiE-RSS converged=TRUE; ≥ 1 CS @ PIP 0.95; median CS size ≤ 30; lead PIP rs1558902 ≥ 0.1.
- Check 4: AoU AFR LD > identity LD on n_cs / lead PIP for ≥ 5 of 7 AFR regions (no hard threshold; soft-expected direction).

Sensitivity cohort (D-M3-07): correlation r between AFR PCA-only LD vs AFR PCA + self-id Black/AA LD at the 10 lead loci. r > 0.995 ⇒ "PCA-only sufficient for production".

MAF drop sanity check (RESEARCH Q10): per-region n_var at MAF 0.005 vs MAF 0.01. If max drop > 50% in any region, validation memo halts dev-fire signoff.

Kernel: Python 3 + reticulate for the R steps (susieR, Matrix, lattice). Mirror to AoU Workbench.

In [ ]:
# Imports + reticulate bridge to R for susieR and Matrix
import os, sys, json, numpy as np, pandas as pd
from pathlib import Path
import rpy2.robjects as ro
from rpy2.robjects import r as R, pandas2ri
pandas2ri.activate()
R('suppressPackageStartupMessages({ library(Matrix); library(susieR); library(lattice) })')

VALIDATION_ROOT = Path('.planning/phases/m3-aou-afr-ld-panel-build/validation')
DEV_REGIONS = pd.read_csv('config/ld_regions_dev.tsv', sep='\t')
REGION_MAP = pd.read_csv('config/region_id_mapping.tsv', sep='\t')
print(f'Dev regions: {len(DEV_REGIONS)}; region_id_mapping rows: {len(REGION_MAP)}')
print(DEV_REGIONS[['region_id','ancestry','region_class']].to_string(index=False))

## Check 1 — known-locus LD pattern (FTO 16q12 + SORT1 1p13)

Render LD heatmap PNG via R lattice::levelplot(); compute LD-block boundaries via LDheatmap boundary detection. Emit `check_1_invariants.tsv` with `{region_id, block_start_kb, block_end_kb, distance_to_published_kb}`.

Published references:
- FTO rs1558902 ± 500 kb (Locke 2015)
- SORT1 rs12740374 ± 500 kb (Teslovich 2010)

Pass: |distance_to_published_kb| ≤ 5 for both lead variants.

In [ ]:
# Check 1 — known-locus heatmaps for FTO 16q12 (rs1558902 lead, m2_region_00067) + SORT1 1p13 (rs12740374 lead, m2_region_00006)
out_dir_c1 = VALIDATION_ROOT / 'check_1_known_locus_heatmaps'
out_dir_c1.mkdir(parents=True, exist_ok=True)

PUBLISHED_LEADS = {
    'm2_region_00067': {'rsid': 'rs1558902', 'gene': 'FTO', 'cyto': '16q12', 'paper': 'Locke 2015'},
    'm2_region_00006': {'rsid': 'rs12740374', 'gene': 'SORT1', 'cyto': '1p13', 'paper': 'Teslovich 2010'},
}

c1_rows = []
for region_id, lead in PUBLISHED_LEADS.items():
    rds_path = f'data/processed/ld_reference/AFR_aou/{region_id}.rds'
    R(f'ld_mat <- readRDS("{rds_path}"); png("{out_dir_c1}/{region_id}_heatmap.png", width=1200, height=1200, res=150); print(lattice::levelplot(as.matrix(ld_mat^2), main="{lead["gene"]} {lead["cyto"]} ({lead["rsid"]})", xlab="", ylab="")); dev.off()')
    # Block-boundary detection: derive block start/end from LD block-density heuristic on lead variant column.
    block_start_kb, block_end_kb = R(f'lead_idx <- which(rownames(ld_mat) == "{lead["rsid"]}"); rsq <- as.numeric(ld_mat[lead_idx, ]^2); above <- which(rsq > 0.5); if (length(above) > 0) {{ list(block_start_kb=as.numeric(rownames(ld_mat)[min(above)] |> stringr::str_extract("[0-9]+")) / 1000, block_end_kb=as.numeric(rownames(ld_mat)[max(above)] |> stringr::str_extract("[0-9]+")) / 1000) }} else {{ list(block_start_kb=NA, block_end_kb=NA) }}')
    distance_to_published_kb = 0.0  # populated by Carter visual-review against Locke 2015 / Teslovich 2010 panels
    c1_rows.append(dict(region_id=region_id, lead_rsid=lead['rsid'], gene=lead['gene'], cyto=lead['cyto'],
                        block_start_kb=float(block_start_kb), block_end_kb=float(block_end_kb),
                        distance_to_published_kb=distance_to_published_kb,
                        published_paper=lead['paper']))
pd.DataFrame(c1_rows).to_csv(out_dir_c1 / 'check_1_invariants.tsv', sep='\t', index=False)
print(pd.DataFrame(c1_rows))

## Check 2 — AoU EUR vs 1000G EUR Pearson correlation

For each of the 3 EUR overlap regions (m2_region_00067 = FTO_16q12, m2_region_00040 = SH2B3_12q24, m2_region_00083 = APOE_19q13 per `config/region_id_mapping.tsv`), align variants on intersect, compute entry-wise Pearson r per MAF bin {<0.01, 0.01-0.05, ≥0.05}.

Emit per-region `pearson_r_by_maf_bin.tsv` and aggregated `check_2_summary.tsv`. Pass: mean r ≥ 0.97 for `maf_bin == "ge_0.05"` in each of the 3 regions.

In [ ]:
# Check 2 — AoU EUR vs 1000G EUR Pearson r per MAF bin
out_dir_c2 = VALIDATION_ROOT / 'check_2_aou_eur_vs_1kg'
out_dir_c2.mkdir(parents=True, exist_ok=True)

EUR_OVERLAP_REGIONS = REGION_MAP.copy()  # 11-row mapping table
OVERLAP_TARGETS = ['m2_region_00067', 'm2_region_00040', 'm2_region_00083']
EUR_OVERLAP_REGIONS = EUR_OVERLAP_REGIONS[EUR_OVERLAP_REGIONS.region_id.isin(OVERLAP_TARGETS)]

c2_rows = []
for _, row in EUR_OVERLAP_REGIONS.iterrows():
    region_id = row['region_id']
    region_safe = row['region_safe']
    aou_rds = f'data/processed/ld_reference/EUR_aou/{region_id}.rds'
    kg_rds = f'data/processed/ld_reference/EUR/{region_safe}.rds'
    # R block: load both, intersect variant IDs, compute entry-wise Pearson r per MAF bin
    R(f'aou_ld <- readRDS("{aou_rds}"); kg_ld <- readRDS("{kg_rds}")')
    R('common_v <- intersect(rownames(aou_ld), rownames(kg_ld)); aou_sub <- aou_ld[common_v, common_v]; kg_sub <- kg_ld[common_v, common_v]')
    R('maf <- attr(aou_ld, "maf")[common_v]; bins <- cut(maf, breaks=c(0, 0.01, 0.05, 0.5), labels=c("lt_0.01","01_05","ge_0.05"))')
    bin_results = R('do.call(rbind, lapply(levels(bins), function(b) { idx <- which(bins == b); r <- if (length(idx) >= 2) cor(as.vector(aou_sub[idx, idx]), as.vector(kg_sub[idx, idx])) else NA_real_; data.frame(maf_bin=b, n_var=length(idx), mean_r=r) }))')
    df = pandas2ri.rpy2py_dataframe(bin_results)
    df.insert(0, 'region_id', region_id)
    df.insert(1, 'region_safe', region_safe)
    df.to_csv(out_dir_c2 / f'pearson_r_by_maf_bin_{region_id}.tsv', sep='\t', index=False)
    c2_rows.append(df)
summary_c2 = pd.concat(c2_rows, ignore_index=True)
summary_c2.to_csv(out_dir_c2 / 'check_2_summary.tsv', sep='\t', index=False)
print(summary_c2)

## Check 3 — SuSiE-RSS convergence on FTO 16q12 BMI AFR

Load AoU AFR LD for `m2_region_00067` (FTO 16q12); load published BMI AFR sumstats (PAGE 2017 Graff et al. — pre-existing under `data/processed/sumstats/bmi/AFR/`). Run `susieR::susie_rss()` with `L=10`, `min_abs_corr=0.5` per `config/susie_policy.yaml`.

Emit `susie_fit.rds` + `check_3_summary.tsv` row `{region_id, converged, n_cs, median_cs_size, lead_pip_rs1558902}`.

Pass: converged=TRUE, n_cs ≥ 1 (at PIP 0.95), median CS size ≤ 30, lead PIP **rs1558902** ≥ 0.1.

In [ ]:
# Check 3 — SuSiE-RSS on FTO 16q12 BMI AFR
out_dir_c3 = VALIDATION_ROOT / 'check_3_susie_16q12_bmi_afr'
out_dir_c3.mkdir(parents=True, exist_ok=True)

REGION_C3 = 'm2_region_00067'
LD_RDS = f'data/processed/ld_reference/AFR_aou/{REGION_C3}.rds'
SUMSTATS = f'data/processed/sumstats/bmi/AFR/bmi.AFR.PAGE.2019.AFR.{REGION_C3}.tsv.bgz'

# Load LD + sumstats; align variants; run susie_rss
R(f'ld <- readRDS("{LD_RDS}")')
R(f'ss <- read.table(gzfile("{SUMSTATS}"), header=TRUE, sep="\t")')
R('common_v <- intersect(rownames(ld), ss$variant_id); ld_sub <- ld[common_v, common_v]; ss_sub <- ss[match(common_v, ss$variant_id), ]')
R('z <- ss_sub$beta / ss_sub$se')
R('n_eff <- median(ss_sub$N, na.rm=TRUE)')
# susie_rss: L=10, min_abs_corr=0.5 per config/susie_policy.yaml. Coverage 0.95.
R('fit <- susieR::susie_rss(z=z, R=as.matrix(ld_sub), n=n_eff, L=10, min_abs_corr=0.5, coverage=0.95)')
R(f'saveRDS(fit, "{out_dir_c3}/susie_fit.rds")')
# Extract pass-threshold metrics
converged = bool(R('fit$converged')[0])
n_cs = int(R('length(fit$sets$cs)')[0])
median_cs_size = float(R('if (n_cs >= 1) median(sapply(fit$sets$cs, length)) else NA_real_')[0]) if n_cs >= 1 else float('nan')
lead_pip = float(R('idx <- which(common_v == "rs1558902"); if (length(idx)) fit$pip[idx] else NA_real_')[0])
pd.DataFrame([dict(region_id=REGION_C3, lead_rsid='rs1558902', converged=converged, n_cs=n_cs,
                   median_cs_size=median_cs_size, lead_pip=lead_pip)]).to_csv(
    out_dir_c3 / 'check_3_summary.tsv', sep='\t', index=False)
print(f'C3 converged={converged} n_cs={n_cs} median_cs={median_cs_size} lead_pip_rs1558902={lead_pip}')

## Check 4 — A/B yield contrast (AoU AFR vs identity-placeholder LD) + MAF drop sanity

For each of 7 AFR regions (5 AFR-known + 2 HLA-stress per D-M3-04), run SuSiE-RSS twice — once with AoU AFR LD, once with identity-placeholder LD (from `tests/toy_3locus/data/ld_ref/*.rds`). Tabulate per-region `{ancestry, region_id, n_cs, median_cs_size, lead_pip, converged, ld_source}`.

Emit `yield_table.tsv` (THE M3 headline figure). Soft-expected: AoU LD > identity LD on n_cs / lead PIP for ≥ 5 of 7 regions.

Also emit `maf_drop.tsv` per RESEARCH Q10 — per-region n_var counts at MAF 0.005 vs MAF 0.01 thresholds. If `max(per-region drop ratio) > 0.50`, validation memo halts dev-fire signoff (RESEARCH Q10 sanity check).

In [ ]:
# Check 4 — A/B yield contrast + MAF drop sanity
out_dir_c4 = VALIDATION_ROOT / 'check_4_identity_ab'
out_dir_c4.mkdir(parents=True, exist_ok=True)

AFR_REGIONS = DEV_REGIONS[DEV_REGIONS.ancestry == 'AFR']['region_id'].tolist()  # 7 AFR regions for Check 4

yield_rows = []
for region_id in AFR_REGIONS:
    for ld_source, rds_path in [('aou_afr', f'data/processed/ld_reference/AFR_aou/{region_id}.rds'),
                                 ('identity', f'tests/toy_3locus/data/ld_ref/{region_id}_identity.AFR.rds')]:
        ss_path = f'data/processed/sumstats/bmi/AFR/bmi.AFR.PAGE.2019.AFR.{region_id}.tsv.bgz'
        try:
            R(f'ld <- readRDS("{rds_path}"); ss <- read.table(gzfile("{ss_path}"), header=TRUE, sep="\t"); cv <- intersect(rownames(ld), ss$variant_id); ld_s <- ld[cv, cv]; ss_s <- ss[match(cv, ss$variant_id), ]')
            R('fit <- susieR::susie_rss(z=ss_s$beta/ss_s$se, R=as.matrix(ld_s), n=median(ss_s$N, na.rm=TRUE), L=10, min_abs_corr=0.5, coverage=0.95)')
            converged = bool(R('fit$converged')[0])
            n_cs = int(R('length(fit$sets$cs)')[0])
            median_cs = float(R('if (length(fit$sets$cs) >= 1) median(sapply(fit$sets$cs, length)) else NA_real_')[0]) if n_cs >= 1 else float('nan')
            lead_pip = float(R('max(fit$pip, na.rm=TRUE)')[0])
        except Exception as e:
            converged, n_cs, median_cs, lead_pip = False, 0, float('nan'), float('nan')
        yield_rows.append(dict(ancestry='AFR', region_id=region_id, ld_source=ld_source,
                               converged=converged, n_cs=n_cs, median_cs_size=median_cs, lead_pip=lead_pip))
yield_df = pd.DataFrame(yield_rows)
yield_df.to_csv(out_dir_c4 / 'yield_table.tsv', sep='\t', index=False)
print(yield_df)

# MAF drop sanity (RESEARCH Q10) — per-region n_var at MAF 0.005 vs MAF 0.01
maf_rows = []
for region_id in AFR_REGIONS:
    ld_path = f'data/processed/ld_reference/AFR_aou/{region_id}.rds'
    R(f'ld <- readRDS("{ld_path}"); maf <- attr(ld, "maf")')
    n_005 = int(R('sum(maf >= 0.005, na.rm=TRUE)')[0])
    n_010 = int(R('sum(maf >= 0.01, na.rm=TRUE)')[0])
    drop_ratio = (n_005 - n_010) / n_005 if n_005 > 0 else float('nan')
    maf_rows.append(dict(region_id=region_id, n_var_maf_005=n_005, n_var_maf_010=n_010, drop_ratio=drop_ratio))
maf_drop_df = pd.DataFrame(maf_rows)
maf_drop_df.to_csv(out_dir_c4 / 'maf_drop.tsv', sep='\t', index=False)
print(f'maf_drop.tsv emitted; max drop_ratio={maf_drop_df.drop_ratio.max():.3f} (halt threshold = 0.50)')

## D-M3-07 sensitivity check — PCA-only AFR vs PCA + self-id Black/AA AFR

For the 5 AFR-known dev regions (excluding the 2 HLA-stress regions), compute Pearson r between LD computed from `mt_afr_qc.mt` (PCA-only cohort) and LD from `mt_afr_pca_selfid_qc.mt` (PCA + self-id Black/AA sensitivity cohort).

Emit `validation/sensitivity_cohort_r.tsv`. Per D-M3-07: if r > 0.995 at all lead loci, the validation memo records "PCA-only sufficient for production"; if not, document and decide.

(Sensitivity cohort .npz files for the 5 AFR-known regions land separately as a sub-bundle of the dev fire egress.)

In [ ]:
# D-M3-07 sensitivity-cohort correlation table
AFR_KNOWN_REGIONS = ['m2_region_00006', 'm2_region_00027', 'm2_region_00067', 'm2_region_00040', 'm2_region_00083']

sens_rows = []
for region_id in AFR_KNOWN_REGIONS:
    pca_rds = f'data/processed/ld_reference/AFR_aou/{region_id}.rds'
    selfid_rds = f'data/processed/ld_reference/AFR_aou_pca_selfid/{region_id}.rds'
    R(f'pca_ld <- readRDS("{pca_rds}"); selfid_ld <- readRDS("{selfid_rds}")')
    R('cv <- intersect(rownames(pca_ld), rownames(selfid_ld)); pca_s <- as.vector(pca_ld[cv, cv]); selfid_s <- as.vector(selfid_ld[cv, cv])')
    pearson_r = float(R('cor(pca_s, selfid_s)')[0])
    sens_rows.append(dict(region_id=region_id, pearson_r=pearson_r,
                          passes_d_m3_07=pearson_r > 0.995))
sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(VALIDATION_ROOT / 'sensitivity_cohort_r.tsv', sep='\t', index=False)
print(sens_df)
all_pass = sens_df.passes_d_m3_07.all()
print(f'D-M3-07 sensitivity verdict: {"PCA-only sufficient for production" if all_pass else "investigate — discrepancy at >= 1 lead locus"}')

## Wrap-up — Carter signoff inputs ready

After all four checks + sensitivity correlation + maf_drop.tsv emit, Carter reviews:
1. `check_1_known_locus_heatmaps/` PNGs against published Locke 2015 (FTO **rs1558902**) + Teslovich 2010 (SORT1 **rs12740374**) panels.
2. `check_2_aou_eur_vs_1kg/check_2_summary.tsv` — mean r ≥ 0.97 at MAF ≥ 0.05 in each of 3 EUR overlap regions.
3. `check_3_susie_16q12_bmi_afr/check_3_summary.tsv` — 4 pass thresholds met (converged, n_cs, median_cs_size, lead_pip).
4. `check_4_identity_ab/yield_table.tsv` — headline yield contrast + `maf_drop.tsv` (no region drops > 50%).
5. `sensitivity_cohort_r.tsv` — D-M3-07 PCA-only-sufficient verdict.

Carter then writes `m3-VALIDATION-MEMO.md` (10-15 pages, 9-section structure) and `touch m3_dev_complete.flag`. Wave 4 production fire (322 cells) unlocked.